# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

> NOTE: DO NOT RUN THESE CELLS IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY

In [5]:
#!pip install -qU ragas==0.2.10

In [6]:
#!pip install -qU langchain-community==0.3.14 langchain-openai==0.2.14 unstructured==0.16.12 langgraph==0.2.61 langchain-qdrant==0.2.0

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

fixes an issue that can happen with mac where you don't have tokenizer

In [7]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/terellbrown/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/terellbrown/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [8]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [9]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [10]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Loan Data use-case!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [11]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [12]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [13]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [14]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs[:20]:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 20, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [15]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node '0ce3bd'. Skipping!
Property 'summary' already exists in node 'bd8745'. Skipping!
Property 'summary' already exists in node 'caba9e'. Skipping!
Property 'summary' already exists in node '8a6a74'. Skipping!
Property 'summary' already exists in node '403170'. Skipping!
Property 'summary' already exists in node '54409b'. Skipping!
Property 'summary' already exists in node '8ca5d8'. Skipping!
Property 'summary' already exists in node 'cdc201'. Skipping!
Property 'summary' already exists in node '41aff5'. Skipping!
Property 'summary' already exists in node 'f8bcf4'. Skipping!
Property 'summary' already exists in node '1492f0'. Skipping!
Property 'summary' already exists in node 'c43047'. Skipping!
Property 'summary' already exists in node '3bc191'. Skipping!
Property 'summary' already exists in node '1ca01e'. Skipping!


Applying CustomNodeFilter:   0%|          | 0/6 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/43 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node '0ce3bd'. Skipping!
Property 'summary_embedding' already exists in node 'bd8745'. Skipping!
Property 'summary_embedding' already exists in node '54409b'. Skipping!
Property 'summary_embedding' already exists in node '8a6a74'. Skipping!
Property 'summary_embedding' already exists in node 'f8bcf4'. Skipping!
Property 'summary_embedding' already exists in node 'caba9e'. Skipping!
Property 'summary_embedding' already exists in node '41aff5'. Skipping!
Property 'summary_embedding' already exists in node '8ca5d8'. Skipping!
Property 'summary_embedding' already exists in node '3bc191'. Skipping!
Property 'summary_embedding' already exists in node '403170'. Skipping!
Property 'summary_embedding' already exists in node '1492f0'. Skipping!
Property 'summary_embedding' already exists in node 'cdc201'. Skipping!
Property 'summary_embedding' already exists in node '1ca01e'. Skipping!
Property 'summary_embedding' already exists in node 'c43047'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 40, relationships: 477)

We can save and load our knowledge graphs as follows.

In [16]:
kg.save("loan_data_kg.json")
loan_data_kg = KnowledgeGraph.load("loan_data_kg.json")
loan_data_kg

KnowledgeGraph(nodes: 40, relationships: 477)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [17]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=loan_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [18]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

### ✅ ANSWER
#### Single Hop Specific Query Synthesizer:
This synthesizer generates straightforward, direct questions that can be answered using a single piece of information from a single document chunks in the knowledge base. These are typically fact-based or specific questions that don't require combining multiple pieces of information. (Example: "When was Bob Marley born?")

#### Multi Hop Abstract Query Synthesizer:
This synthesizer creates more complex questions that require synthesizing information from multiple document chunks to form abstract or high-level conclusions. It focuses on generating questions that test the system's ability to understand broader concepts and relationships between different pieces of information. Focuses less on specific properties shared between document chunks and more on conceptual relationships (likely matches on embedding). (Example: "What technological and economic breakthroughs led to the mass adoption of LLM?")

#### Multi Hop Specific Query Synthesizer:
While this does use multiple question-answer pairs like the abstract synthesizer, it differs in that it creates specific, detailed questions that require connecting multiple concrete facts from multiple document chunks rather than abstract concepts. These questions test the system's ability to combine specific pieces of information from different sources to provide a precise answer. (Example: "What is the relationship between {concept from chunk A} and {concept from chunk B}?")


Finally, we can use our `TestSetGenerator` to generate our testset!

In [19]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,What is the significance of Volume 2 in the c...,"[Chapter 1 Academic Years, Academic Calendars,...",The provided context does not specify the cont...,single_hop_specifc_query_synthesizer
1,What does 34 CFR 668.3(a) specify regarding ac...,[Regulatory Citations Academic year minimums: ...,34 CFR 668.3(a) pertains to the minimum number...,single_hop_specifc_query_synthesizer
2,Under what conditions is clinical work include...,[Inclusion of Clinical Work in a Standard Term...,Inclusion of clinical work in a standard term ...,single_hop_specifc_query_synthesizer
3,FWS is what kind of program and how does it di...,[Non-Term Characteristics A program that measu...,The Federal Work-Study (FWS) Program is an exc...,single_hop_specifc_query_synthesizer
4,How does the Direct Loan program determine dis...,[both the credit or clock hours and the weeks ...,The Direct Loan program's disbursement timing ...,single_hop_specifc_query_synthesizer
5,How does credit hour allocation for clinical e...,[<1-hop>\n\nInclusion of Clinical Work in a St...,Credit hours associated with clinical experien...,multi_hop_abstract_query_synthesizer
6,How do the number of weeks of instructional ti...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",The academic year must include a minimum numbe...,multi_hop_abstract_query_synthesizer
7,How does eligibilty calcuation based on hours ...,[<1-hop>\n\nboth the credit or clock hours and...,The eligibilty calcuation based on hours and w...,multi_hop_abstract_query_synthesizer
8,Volume 8 and Volume 8 how does that affect dis...,[<1-hop>\n\nboth the credit or clock hours and...,The context explains that the scheduled paymen...,multi_hop_specific_query_synthesizer
9,Whay do Chapter 2 and Chapter 3 both rel8 to c...,[<1-hop>\n\nInclusion of Clinical Work in a St...,Chapter 2 discusses the inclusion of clinical ...,multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [20]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs[:20], testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node '6fd129'. Skipping!
Property 'summary' already exists in node '5a7ffd'. Skipping!
Property 'summary' already exists in node 'd0d599'. Skipping!
Property 'summary' already exists in node 'd5a719'. Skipping!
Property 'summary' already exists in node 'a39664'. Skipping!
Property 'summary' already exists in node 'e5a7c7'. Skipping!
Property 'summary' already exists in node '62eb3b'. Skipping!
Property 'summary' already exists in node 'ff5f4c'. Skipping!
Property 'summary' already exists in node '2df6fe'. Skipping!
Property 'summary' already exists in node 'b56c68'. Skipping!
Property 'summary' already exists in node 'ed6f94'. Skipping!
Property 'summary' already exists in node 'f3b45d'. Skipping!
Property 'summary' already exists in node 'da624d'. Skipping!
Property 'summary' already exists in node '9b0d71'. Skipping!


...

Property 'summary_embedding' already exists in node '6fd129'. Skipping!
Property 'summary_embedding' already exists in node 'd0d599'. Skipping!
Property 'summary_embedding' already exists in node '2df6fe'. Skipping!
Property 'summary_embedding' already exists in node 'd5a719'. Skipping!
Property 'summary_embedding' already exists in node '62eb3b'. Skipping!
Property 'summary_embedding' already exists in node 'a39664'. Skipping!
Property 'summary_embedding' already exists in node '5a7ffd'. Skipping!
Property 'summary_embedding' already exists in node 'ed6f94'. Skipping!
Property 'summary_embedding' already exists in node 'ff5f4c'. Skipping!
Property 'summary_embedding' already exists in node '9b0d71'. Skipping!
Property 'summary_embedding' already exists in node 'f3b45d'. Skipping!
Property 'summary_embedding' already exists in node 'e5a7c7'. Skipping!
Property 'summary_embedding' already exists in node 'da624d'. Skipping!
Property 'summary_embedding' already exists in node 'b56c68'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [21]:
import pandas as pd
pd.set_option('display.max_colwidth', None)  # Show full column content
pd.set_option('display.max_rows', None)      # Show all rows
pd.set_option('display.max_columns', None)   # Show all columns
pd.set_option('display.width', None)         # Auto-detect display width

In [22]:
dataset.to_pandas()

user_input  \
0                                                                                                                                              What information can I find in the Knowledge Center regarding academic year requirements and instructional time for Title IV programs?   
1                                                                                                                                                                                            What does 34 CFR 668.3(a) mean like in the rules about academic year minimums and stuff?   
2                                                              How does the inclusion of clinical work in a standard term affect the classification of the term for federal aid purposes, and what criteria must be met for such clinical work to be included within a standard term?   
3                                                                                                                                                                                                Is the Federal Work-Study (FWS) program subject to payment periods for disbursement?   
4                                                                                                                                                                                     Wht are the payment perods based on the weeks of instrucional time for diffrent acedemic years?   
5    How do the disbursement timing requirements for federal student aid differ between clock-hour or non-term credit-hour programs and subscription-based programs, especially considering the impact of accelerated progression and multiple disbursements within a payment period?   
6                                                                                                                                                        so academic years and regulatory citations like 34 CFR 668.3(a) and (b) tell us about weeks of instruction and stuff, right?   
7                                                                                                                                                                                           how disbursmnt timing in subscriptn-based programs and how it affect aid disbursmnt reqs?   
8                       How do Appendix A and Appendix B relate to the disbursement timing and eligibility requirements for students in clock-hour, non-term credit-hour, and subscription-based programs, particularly regarding accelerated progression and multiple disbursements?   
9   How do Volume 2 and Volume 8 together inform the requirements for disbursement timing and academic year definitions in clock-hour and credit-hour programs for federal student aid, particularly regarding the impact of accelerated progression on disbursement and loan limits?   
10                                                                                                                                                                                                                                             Volume 8 and Volume 7 are related how?   
11                                                                                                                                 How do the guidelines in Volume 2 and Volume 8 relate to the definition of academic years and clinical work inclusion for Title IV aid compliance?   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [25]:
from langsmith import Client

client = Client()

dataset_name = "Loan Synthetic Data 3"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Loan Synthetic Data"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [26]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [27]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [28]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [29]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [30]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan RAG"
)

In [31]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [32]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

For our LLM, we will be using TogetherAI's endpoints as well!

We're going to be using Meta Llama 3.1 70B Instruct Turbo - a powerful model which should get us powerful results!

In [33]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [34]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [35]:
rag_chain.invoke({"question" : "What kinds of loans are available?"})

'The kinds of loans available include:\n\n- Direct Subsidized Loans  \n- Direct Unsubsidized Loans  \n- Direct PLUS Loans (including student Federal PLUS Loans and parent Direct PLUS Loans)  \n- Subsidized Federal Stafford Loans  \n- Unsubsidized Federal Stafford Loans  \n- Federal SLS Loans  \n- Federal PLUS Loans (made under the Federal Family Education Loan (FFEL) Program before July 1, 2010)\n\nNote: New FFEL Program loans have not been made since June 30, 2010. Graduate or professional students are eligible only for Direct Unsubsidized Loans, not Direct Subsidized Loans.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [36]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [37]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

empathy_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "empathy": "Is this response empathetic? Does it make the user feel like they are being heard?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

### ✅ ANSWER
- `qa_evaluator`: based on the judgment of the LLM does the response accurately answer the question
- `labeled_helpfulness_evaluator`: Compares the RAG app output to the reference answer to determine if the output accurately answer the question
- `empathy_evaluator`: does the application respond with a level of empathy that matches the level of inconvenience and fustration that the user is experiencing

## LangSmith Evaluation

In [38]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'passionate-cheese-18' at:
https://smith.langchain.com/o/c2cfcbd8-d5df-509f-8f0e-973ec8ab5a6b/datasets/23e40814-8616-41c9-9dbb-b496c09c147c/compare?selectedSessions=565a4722-9e6a-44e8-936e-9a1984b9d5b7




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,How do the guidelines in Volume 2 and Volume 8 relate to the definition of academic years and clinical work inclusion for Title IV aid compliance?,I don't know.,None,"Volume 2 outlines the requirements for academic years, including minimum instructional weeks and the treatment of standard and nonstandard terms, while Volume 8 provides guidance on including clinical work in standard term periods. Together, they clarify that clinical work meeting specific criteria can be included within standard terms, which must align with the academic year definitions established in Volume 2, ensuring compliance with Title IV aid disbursement and program requirements.",0,0,0,0.937130,a0632391-a8ab-4435-b5d2-48ef2b82b891,45a0dbf4-7d2b-4977-8849-8cadcf05c0c7
1,Volume 8 and Volume 7 are related how?,"Based on the provided context, Volume 7 covers the Federal Pell Grant Program, providing information to assist schools in determining student eligibility and calculating Pell Grant amounts. Volume 8 covers the Direct Loan Program, providing information to assist schools in determining eligibility for Direct Loans, counseling borrowers, and awarding Direct Loans.\n\nBoth volumes are parts of the Federal Student Aid Handbook and serve to guide schools in administering different types of federal student aid programs. Volume 7 focuses on Pell Grants, which are grants (not loans), while Volume 8 focuses on Direct Loans.\n\nTherefore, Volume 7 and Volume 8 are related in that they are both parts of the Federal Student Aid Handbook and provide complementary information for administering different federal student aid programs: Pell Grants in Volume 7 and Direct Loans in Volume 8.",None,"Volume 8 discusses the inclusion of clinical work in standard term periods and provides guidance on nonstandard terms and disbursement timing for federal student aid programs, while Volume 7 covers calculations for Pell Grant and TEACH Grant awards based on program hours and weeks. Together, they provide comprehensive information on how program structure and clinical work impact disbursement and award calculations in federal student aid programs.",0,0,0,3.362168,c7469d3d-5e06-453f-93d9-e425084bf517,3cd5cb33-cb6a-4546-a616-caddc1f31380
2,"How do Volume 2 and Volume 8 together inform the requirements for disbursement timing and academic year definitions in clock-hour and credit-hour programs for federal student aid, particularly regarding the impact of accelerated progression on disbursement and loan limits?","Based on the provided context:\n\n- Volume 2 outlines requirements for disbursement timing in clock-hour and credit-hour programs, including the need to consider both credit or clock hours and weeks of instructional time in a payment period before disbursing Pell Grant, FSEOG, TEACH Grant, or Direct Loan funds. It also discusses the impact of accelerated progression on disbursement timing, particularly in clock-hour or non-term credit-hour programs and for Direct Loan programs with nonstandard terms.\n\n- Volume 8 provides detailed guidance on Direct Loan proration, annual loan limit progression, and recalculation in clock-hour and non-term credit-hour programs. It includes examples illustrating how prorated annual loan limits must be determined or re-prorated when students accelerate progress or when program lengths differ from a standard academic year. Volume 8 also covers how loans must be adjusted when a student completes portions of the academic year early or in a nonstandard manner.\n\nTogether, these volumes inform that:\n\n1. Disbursement timing for federal aid in clock-hour and credit-hour programs must account for both the number of credit/clock hours completed and weeks of instructional time, especially before disbursing funds for the next payment period.\n\n2. Accelerated progression (completing credit/clock hours fas

## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [39]:
EMPATHY_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

You must answer the question using empathy and kindness, and make sure the user feels heard.

Context: {context}
Question: {question}
"""

empathy_rag_prompt = ChatPromptTemplate.from_template(EMPATHY_RAG_PROMPT)

In [40]:
rag_documents = docs

In [41]:
from langchain.text_splitter import RecursiveCharacterTextSplitter


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

if we know the chunking process we are using does not cut off context (only produces paragraphs for example) then no need for overlap

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

#### ✅ ANSWER
Increasing the chunk size means more context and semantic meaning is captured in each chunk which leads to:
- when chunks are returned there is more context available to help you answer the question which could lead to better answers the question
- when applying similarity search it may be difficult to match a very specific to the appropriate chunk because the part of the chunk that relates to the query has less influence on the semanting meaning of the chunk

Increasing the chunk size means less chunks are created which leads to:
- fewer chunks that need to be compared during similarity search and lower latency
- less storage capacity consumed

In [42]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

#### ✅ ANSWER
A larger embedding model, means more semantic meaning can be captured in the generated vector which leads to:
- a chunk can be matched to a query with greater semantic granularity => improved retrieval accuracy => improved accuracy of answer
- consumes more time and resources to generate embeddings and perform semantic search => increased latency and cost => worst user experience, less profit

In [43]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan Data for RAG"
)

In [44]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [45]:
empathy_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | empathy_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [46]:
empathy_rag_chain.invoke({"question" : "What kinds of loans are available?"})

"Thank you for your question! Based on the information provided, there are several kinds of loans available to students and parents to help with educational costs:\n\n1. **Direct Subsidized Loans** – These are loans where the government pays the interest while the student is in school, based on financial need.\n\n2. **Direct Unsubsidized Loans** – Available to both dependent and independent students, these loans do not require demonstrating financial need, and interest accrues while in school.\n\n3. **Direct PLUS Loans** – These are loans parents of dependent students can take out to cover education costs, assuming they meet eligibility requirements. There is no fixed loan limit for PLUS Loans, but the amount cannot exceed the student's cost of attendance minus other financial aid.\n\nAdditionally, if a parent cannot obtain a Direct PLUS Loan, the dependent student may qualify for increased amounts of Direct Unsubsidized Loans to help cover costs.\n\nIt’s clear from your question that 

Finally, we can evaluate the new chain on the same test set!

In [47]:
evaluate(
    empathy_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "empathy_rag_chain"},
)

View the evaluation results for experiment: 'cold-sky-90' at:
https://smith.langchain.com/o/c2cfcbd8-d5df-509f-8f0e-973ec8ab5a6b/datasets/23e40814-8616-41c9-9dbb-b496c09c147c/compare?selectedSessions=441b252d-9f1c-4fe4-91d6-04203a4a5b5c




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,How do the guidelines in Volume 2 and Volume 8 relate to the definition of academic years and clinical work inclusion for Title IV aid compliance?,"Thank you for your thoughtful question. I can see you're looking to understand how the guidelines in Volume 2 and Volume 8 connect to defining academic years and including clinical work for Title IV aid compliance.\n\nBased on the context provided:\n\n- Volume 2, Chapter 2 is mentioned as having more details on requirements for eligible programs that are shorter than an academic year, which relates to defining academic years for Title IV purposes.\n\n- Volume 3, Chapter 1 of the FSA Handbook (though not Volume 8) discusses inclusion of clinical work in standard or certain nonstandard terms and exceptions to normal loan and disbursement requirements when clinical work is included.\n\nUnfortunately, the information does not specifically mention Volume 8 or directly explain how these two volumes together relate to academic year definitions and clinical work inclusion for Title IV compliance.\n\nI want to acknowledge that understanding the detailed regulations around academic years and Title IV aid can be complex and sometimes the guidelines are spread across various volumes, which can feel overwhelming. I’m sorry I cannot provide a more direct connection based on what’s available here.\n\nIf you have access to the full volumes or can share more details from Volume 2 and Volume 8 specifically, I’d be glad to help further! Meanwhile, please know I’m here to support you as you navigate this information.",None,"Volume 2 outlines the requirements for academic years, including minimum instructional weeks and the treatment of standard and nonstandard terms, while Volume 8 provides guidance on including clinical work in standard term periods. Together, they clarify that clinical work meeting specific criteria can be included within standard terms, which must align with the academic year definitions established in Volume 2, ensuring compliance with Title IV aid disbursement and program requirements.",0,0,1,6.004880,a0632391-a8ab-4435-b5d2-48ef2b82b891,76204bdf-9f70-4a5f-b9f7-d035e6cd13fd
1,Volume 8 and Volume 7 are related how?,"Thank you for your thoughtful question. Based on the context provided, Volume 8 and Volume 7 of the Federal Student Aid Handbook are related in that both volumes offer guidance on different aspects of federal student financial aid programs.\n\nVolume 7 focuses on the Federal Pell Grant Program, providing information to help schools determine student eligibility and calculate grant amounts. Volume 8, on the other hand, centers on the William D. Ford Federal Direct Loan Program, assisting schools in determining eligibility for Direct Loans, counseling borrowers, and awarding those loans.\n\nBoth volumes aim to support schools in administering federal aid programs effectively for the 2025-2026 period, with updated information and changes relevant to their specific aid types. While they cover different programs—grants versus loans—they work together as part of the larger framework to help students finance their education.\n\nI hope this helps clarify how these volumes are connected. If you have any more questions or need further explanation, please feel free to ask. I'm here to assist you!",None,"Volume 8 discusses the inclusion of clinical work in standard term periods and provides guidance on nonstandard terms and disbursement timing for federal student aid programs, while Volume 7 covers calculations for Pell Grant and TEACH Grant awards based on program hours and weeks. Together, they provide comprehensive information on how program structure and clinical work impact disbursement and award calculations in federal student aid programs.",0,0,1,4.917004,c7469d3d-5e06-453f-93d9-e425084bf517,fa75d288-034a-4fc3-b70f-e8e7169b0b3d
2,"

#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.

### ✅ ANSWER


![Experiment comparison showing metrics for correctness, empathy, and helpfulness between two RAG implementations](experiment_comparison.png)







The evaluation results showed:
1. The empathy-focused chain scored higher on empathy metrics (all answers became empathetic)
2. However, it had fewer correct answers compared to the basic chain
3. Both chains maintained similar helpfulness scores

This suggests that while we successfully improved the emotional intelligence of the responses, it came at a slight cost to factual accuracy. This is a common trade-off in RAG systems where adding additional objectives (like empathy) can sometimes reduce performance on the primary objective (accuracy).